In [1]:
import pandas as pd
import numpy as np

In [2]:
ENTREPOT_PATH = '/home/tbadie/Bureau/data/data_entrepot_outils/'
donnees = {}

def import_dfs(df_names, path_data, sep = ','):
    i = 0
    for df_name in df_names: 
        donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, low_memory=False).replace({'\r\n': '\n'}, regex=True)

tables = [
    'noeuds_realise',
    # 'entite_unique_par_sdc_nettoyage', 
    'zone',
    'parcelle'
    ]

# import des données
import_dfs(tables, ENTREPOT_PATH, sep = ',')

In [34]:
nd = donnees['noeuds_realise'][['id','culture_id','zone_id']].rename(columns={'id':'noeuds_realise_id'})
zone = donnees['zone'][['id','surface','parcelle_id']].rename(columns={'id':'zone_id'})
parcelle = donnees['parcelle'][['id','surface','sdc_id']].rename(columns={'id':'parcelle_id', 'surface':'surface_parcelle'})

In [35]:
df = nd.merge(zone, on = 'zone_id', how = 'left').merge(parcelle, on = 'parcelle_id', how = 'left')
df = df.loc[(df['sdc_id'].notna()) & (df['surface'] != 0)]

In [36]:
df['nb_itk_mm_zone'] = df.groupby("zone_id")["noeuds_realise_id"].transform("count")
df['surface_ponderee_zone'] = df['surface'] / df['nb_itk_mm_zone']

df['surface_ponderee_totale'] = df.groupby("sdc_id")["surface_ponderee_zone"].transform("sum")
df["surface_developpee_totale"] = df.groupby("sdc_id")["surface"].transform("sum")

df['poids_surface_ponderee'] = df['surface_ponderee_zone'] / df['surface_ponderee_totale']
df['poids_surface_developpee'] = df['surface'] / df['surface_ponderee_totale']
df['poids_surface_developpee_normalisee'] = df['surface'] / df['surface_developpee_totale']

df = df[['noeuds_realise_id', 'culture_id', 'sdc_id', 'poids_surface_ponderee', 'poids_surface_developpee', 'poids_surface_developpee_normalisee']]

In [50]:
df.loc[df['poids_surface_developpee_normalisee'] < df['poids_surface_ponderee']-0.01].sample(5)

,noeuds_realise_id,culture_id,sdc_id,poids_surface_ponderee,poids_surface_developpee,poids_surface_developpee_normalisee
127775,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_22...,0.038113,0.038113,0.024310
44323,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_20...,0.149701,0.149701,0.127852
1012,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_bd...,0.227142,0.227142,0.168014
38205,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_29...,0.023069,0.023069,0.006982
75458,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_ab...,0.050963,0.050963,0.038838


In [51]:
df.loc[df['poids_surface_developpee_normalisee'] > df['poids_surface_ponderee']+0.01].sample(5)

,noeuds_realise_id,culture_id,sdc_id,poids_surface_ponderee,poids_surface_developpee,poids_surface_developpee_normalisee
55849,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_3b...,0.084370,0.168741,0.144378
7059,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_75...,0.016478,0.032956,0.028953
48015,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_e5...,0.023219,0.046438,0.033486
105579,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_b9...,0.021374,0.042748,0.035462
103495,fr.inra.agrosyst.api.entities.effective.Effect...,fr.inra.agrosyst.api.entities.CroppingPlanEntr...,fr.inra.agrosyst.api.entities.GrowingSystem_b3...,0.022070,0.066209,0.055541


In [49]:
df.groupby("sdc_id")["poids_surface_developpee"].apply("sum").sort_values(ascending=False)

sdc_id
fr.inra.agrosyst.api.entities.GrowingSystem_f3c49758-0a8d-484a-ba10-f8e353af3462    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_159ebcce-5b83-4814-8aab-6dab6c24b414    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_288c3360-04b0-4b39-9bd5-06a45a705110    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_e91191aa-93dc-476e-8821-2fcfeb42c545    7.0
fr.inra.agrosyst.api.entities.GrowingSystem_44277df3-7353-4b81-b993-370a3b1337d9    6.0
                                                                                   ... 
fr.inra.agrosyst.api.entities.GrowingSystem_91c5f1d9-d9ac-442c-b3be-9f02de92a53f    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_93fb16d1-a082-4e18-8df4-2acb48c4f289    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_be7bb970-6569-4058-81db-c9d3b7f46cf8    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_d2e149dd-72f5-43f7-94b2-7ce6655034c0    1.0
fr.inra.agrosyst.api.entities.GrowingSystem_d2615951-0cc8-4210-91e3-c36764d12848    1.0
Name: poids_surface_devel